# Spotify Streaming Audit — Dataset Grammy Awards

## Limpieza y preparación de datos

En esta sección trabajaremos con el dataset adicional de Grammy Awards.

El objetivo es preparar una base limpia, ordenada y reproducible que pueda utilizarse posteriormente para:

- análisis exploratorio,
- cruces con el dataset principal de Spotify,
- SQL,
- visualizaciones,
- dashboards,
- y análisis avanzados del proyecto.

Durante esta limpieza realizaremos:

- carga del archivo,
- revisión inicial,
- limpieza de columnas,
- revisión de nulos,
- eliminación de duplicados,
- conversión de fechas,
- limpieza de texto,
- conservación de URLs y nombres artísticos,
- y exportación del dataset limpio.

In [49]:
# ============================================
# INSTALACIÓN DE LIBRERÍAS NECESARIAS
# ============================================

!pip install duckdb -q

In [50]:
# ============================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================

import pandas as pd
import numpy as np
import re
import os
import zipfile

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import duckdb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

import joblib

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

os.makedirs("data_clean", exist_ok=True)
os.makedirs("visualizations", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## Carga del archivo

El archivo recibido viene comprimido en formato `.zip`.

Primero se sube el archivo a Google Colab y luego se extrae para acceder al archivo `.csv` original.

In [51]:
from google.colab import files

uploaded = files.upload()

Saving Data_spotify (1).zip to Data_spotify (1) (2).zip


In [52]:
# ============================================
# EXTRAER ARCHIVO ZIP
# ============================================

zip_filename = list(uploaded.keys())[0]

extract_folder = "grammy_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Archivo extraído correctamente.")
print("Archivos encontrados:")
print(os.listdir(extract_folder))

Archivo extraído correctamente.
Archivos encontrados:
['the_grammy_awards.csv']


## Lectura del dataset

Después de extraer el archivo comprimido, cargamos el archivo CSV en un DataFrame de pandas.

In [53]:
# ============================================
# CARGA DEL DATASET
# ============================================

csv_files = [
    file for file in os.listdir(extract_folder)
    if file.endswith(".csv")
]

csv_path = os.path.join(extract_folder, csv_files[0])

df_grammy_raw = pd.read_csv(csv_path)

print("Dataset cargado correctamente.")
print("Dimensiones originales:", df_grammy_raw.shape)

df_grammy_raw.head()

Dataset cargado correctamente.
Dimensiones originales: (4810, 10)


,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner
0,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Fi...",https://www.grammy.com/sites/com/files/styles/...,True
1,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin V...",https://www.grammy.com/sites/com/files/styles/...,True
2,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster ...",https://www.grammy.com/sites/com/files/styles/...,True
3,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph H...",https://www.grammy.com/sites/com/files/styles/...,True
4,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar C...",https://www.grammy.com/sites/com/files/styles/...,True


## Exploración inicial

Revisamos la estructura general del dataset:

- número de filas y columnas,
- tipos de datos,
- nombres de columnas,
- valores nulos,
- y posibles duplicados.

Esta etapa permite entender la calidad inicial de la base antes de limpiarla.

In [54]:
df_grammy_raw.shape

(4810, 10)

In [55]:
df_grammy_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4810 entries, 0 to 4809
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          4810 non-null   int64 
 1   title         4810 non-null   object
 2   published_at  4810 non-null   object
 3   updated_at    4810 non-null   object
 4   category      4810 non-null   object
 5   nominee       4804 non-null   object
 6   artist        2970 non-null   object
 7   workers       2620 non-null   object
 8   img           3443 non-null   object
 9   winner        4810 non-null   bool  
dtypes: bool(1), int64(1), object(8)
memory usage: 343.0+ KB


In [56]:
df_grammy_raw.columns

Index(['year', 'title', 'published_at', 'updated_at', 'category', 'nominee',
       'artist', 'workers', 'img', 'winner'],
      dtype='object')

In [57]:
df_grammy_raw.isnull().sum()

,0
year,0
title,0
published_at,0
updated_at,0
category,0
nominee,6
artist,1840
workers,2190
img,1367
winner,0


In [58]:
df_grammy_raw.duplicated().sum()

np.int64(0)

## Limpieza de nombres de columnas

Para trabajar de forma ordenada, transformamos los nombres de columnas a formato `snake_case`.

Esto significa:

- todo en minúsculas,
- sin espacios,
- sin caracteres especiales,
- y usando guiones bajos `_`.

Importante: esta transformación se aplica solo a los nombres de columnas, no a los valores internos del dataset.

In [59]:
# ============================================
# FUNCIÓN PARA CONVERTIR NOMBRES DE COLUMNAS A SNAKE_CASE
# ============================================

def to_snake_case(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name

In [60]:
df_grammy = df_grammy_raw.copy()

df_grammy.columns = [
    to_snake_case(col)
    for col in df_grammy.columns
]

df_grammy.columns

Index(['year', 'title', 'published_at', 'updated_at', 'category', 'nominee',
       'artist', 'workers', 'img', 'winner'],
      dtype='object')

## Revisión de duplicados

Eliminamos filas duplicadas para evitar contar registros repetidos en análisis posteriores.

In [61]:
duplicados_antes = df_grammy.duplicated().sum()

print("Duplicados antes de limpiar:", duplicados_antes)

Duplicados antes de limpiar: 0


In [62]:
df_grammy = df_grammy.drop_duplicates()

duplicados_despues = df_grammy.duplicated().sum()

print("Duplicados después de limpiar:", duplicados_despues)
print("Dimensiones después de eliminar duplicados:", df_grammy.shape)

Duplicados después de limpiar: 0
Dimensiones después de eliminar duplicados: (4810, 10)


## Limpieza de columnas de texto

Las columnas de texto se limpian eliminando:

- espacios innecesarios,
- dobles espacios,
- inconsistencias de formato.

No convertiremos todos los valores a `snake_case`, ya que:

- nombres de artistas perderían legibilidad,
- URLs podrían dañarse,
- categorías y títulos perderían claridad,
- y ciertos nombres artísticos especiales perderían significado.


In [63]:
# ============================================
# IDENTIFICAR COLUMNAS DE FECHA Y TEXTO
# ============================================

date_columns = [
    col for col in df_grammy.columns
    if "date" in col or "published" in col or "updated" in col
]

text_columns = [
    col for col in df_grammy.select_dtypes(include="object").columns
    if col not in date_columns
]

print("Columnas de fecha:", date_columns)
print("Columnas de texto:", list(text_columns))

Columnas de fecha: ['published_at', 'updated_at']
Columnas de texto: ['title', 'category', 'nominee', 'artist', 'workers', 'img']


In [64]:
# ============================================
# LIMPIEZA DE TEXTO SIN DAÑAR NOMBRES NI URLS
# ============================================

for col in text_columns:

    df_grammy[col] = (
        df_grammy[col]
        .where(df_grammy[col].notna(), pd.NA)
    )

    df_grammy[col] = (
        df_grammy[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Limpieza de texto completada correctamente.")

df_grammy[text_columns].head()

Limpieza de texto completada correctamente.


,title,category,nominee,artist,workers,img
0,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Fi...",https://www.grammy.com/sites/com/files/styles/...
1,62nd Annual GRAMMY Awards (2019),Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin V...",https://www.grammy.com/sites/com/files/styles/...
2,62nd Annual GRAMMY Awards (2019),Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster ...",https://www.grammy.com/sites/com/files/styles/...
3,62nd Annual GRAMMY Awards (2019),Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph H...",https://www.grammy.com/sites/com/files/styles/...
4,62nd Annual GRAMMY Awards (2019),Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar C...",https://www.grammy.com/sites/com/files/styles/...


## Conversión de fechas

Las columnas relacionadas con fechas se convierten a formato datetime para facilitar análisis temporales posteriores.

In [65]:
for col in date_columns:
    df_grammy[col] = pd.to_datetime(
        df_grammy[col],
        errors="coerce",
        utc=True
    )

df_grammy[date_columns].head()

,published_at,updated_at
0,2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00
1,2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00
2,2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00
3,2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00
4,2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00


## Revisión final de valores nulos

Después de las transformaciones, revisamos nuevamente los valores nulos.

Esto permite detectar si alguna conversión generó valores faltantes.

In [66]:
null_summary = (
    df_grammy
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

null_summary

,0
workers,2190
artist,1840
img,1367
nominee,6
title,0
year,0
category,0
updated_at,0
published_at,0
winner,0


## Validación de calidad de datos

Antes de finalizar el proceso de limpieza, verificamos:

- estructura final del dataset,
- tipos de datos,
- dimensiones,
- nombres de columnas,
- integridad de URLs,
- y preservación de nombres artísticos.

Esta validación permite asegurar que la base se encuentra lista para análisis posteriores.

In [67]:
df_grammy.head()

,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner
0,2019,62nd Annual GRAMMY Awards (2019),2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Fi...",https://www.grammy.com/sites/com/files/styles/...,True
1,2019,62nd Annual GRAMMY Awards (2019),2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin V...",https://www.grammy.com/sites/com/files/styles/...,True
2,2019,62nd Annual GRAMMY Awards (2019),2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster ...",https://www.grammy.com/sites/com/files/styles/...,True
3,2019,62nd Annual GRAMMY Awards (2019),2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00,Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph H...",https://www.grammy.com/sites/com/files/styles/...,True
4,2019,62nd Annual GRAMMY Awards (2019),2020-05-19 12:10:28+00:00,2020-05-19 12:10:28+00:00,Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar C...",https://www.grammy.com/sites/com/files/styles/...,True


In [68]:
df_grammy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4810 entries, 0 to 4809
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   year          4810 non-null   int64              
 1   title         4810 non-null   string             
 2   published_at  4810 non-null   datetime64[ns, UTC]
 3   updated_at    4810 non-null   datetime64[ns, UTC]
 4   category      4810 non-null   string             
 5   nominee       4804 non-null   string             
 6   artist        2970 non-null   string             
 7   workers       2620 non-null   string             
 8   img           3443 non-null   string             
 9   winner        4810 non-null   bool               
dtypes: bool(1), datetime64[ns, UTC](2), int64(1), string(6)
memory usage: 343.0 KB


In [69]:
print("Dimensiones finales del dataset limpio:")
print(df_grammy.shape)

Dimensiones finales del dataset limpio:
(4810, 10)


In [70]:
df_grammy.columns

Index(['year', 'title', 'published_at', 'updated_at', 'category', 'nominee',
       'artist', 'workers', 'img', 'winner'],
      dtype='object')

In [71]:
# Validación de que URLs y nombres artísticos se conservaron correctamente

df_grammy[['artist', 'category', 'title', 'img']].head()

,artist,category,title,img
0,Billie Eilish,Record Of The Year,62nd Annual GRAMMY Awards (2019),https://www.grammy.com/sites/com/files/styles/...
1,Bon Iver,Record Of The Year,62nd Annual GRAMMY Awards (2019),https://www.grammy.com/sites/com/files/styles/...
2,Ariana Grande,Record Of The Year,62nd Annual GRAMMY Awards (2019),https://www.grammy.com/sites/com/files/styles/...
3,H.E.R.,Record Of The Year,62nd Annual GRAMMY Awards (2019),https://www.grammy.com/sites/com/files/styles/...
4,Khalid,Record Of The Year,62nd Annual GRAMMY Awards (2019),https://www.grammy.com/sites/com/files/styles/...


## Conclusiones de la limpieza de datos

Después del proceso de limpieza y preparación del dataset de Grammy Awards, se obtuvieron las siguientes observaciones:

### Hallazgos principales

- Los nombres de columnas fueron estandarizados a formato `snake_case`.
- Los valores de texto conservaron su formato original para mantener legibilidad.
- Las URLs de imágenes y los nombres artísticos no fueron modificados para preservar la integridad semántica de los datos.
- Se revisaron y eliminaron registros duplicados en caso de existir.
- Las columnas de fecha fueron convertidas correctamente a formato datetime.
- El dataset quedó estructurado y preparado para análisis exploratorio, SQL, dashboards y posibles modelos de Machine Learning.

### Resultado final

La base limpia podrá utilizarse posteriormente para:

- análisis de artistas y categorías,
- cruces con el dataset principal de Spotify,
- análisis temporales,
- dashboards interactivos,
- y análisis avanzados relacionados con premios musicales.

El dataset final quedó almacenado correctamente en formato `.csv`.

### Limitaciones identificadas

- El dataset contiene información histórica y no representa necesariamente el desempeño actual de artistas o categorías.
- La calidad de ciertos campos depende de la información originalmente publicada por la Academia Grammy.

In [72]:
print("=" * 60)
print("RESUMEN FINAL DE LIMPIEZA")
print("=" * 60)

print(f"Filas finales: {df_grammy.shape[0]}")
print(f"Columnas finales: {df_grammy.shape[1]}")
print(f"Duplicados eliminados: {duplicados_antes}")

print()
print("Dataset preparado correctamente para análisis.")
print("=" * 60)

RESUMEN FINAL DE LIMPIEZA
Filas finales: 4810
Columnas finales: 10
Duplicados eliminados: 0

Dataset preparado correctamente para análisis.
